# Optional Colab YOLO train (not required for the demo)

The console already ships `ml/weights/yolov8n_airspace.pt` (val mAP50 **0.3182**).
Do **not** run this notebook to finish the project. Training is stopped.

Self-contained GPU job if someone later wants more epochs. **Do not train on the laptop.**
This is a complete mix-train (VisDrone + AU-AIR train, VisDrone-only val) for the 6-class
detector, default **50 epochs** on Colab GPU.

**Runtime → Change runtime type → T4 GPU** (L4 / A100 also fine) → **Runtime → Run all**.

| Item | Path |
| --- | --- |
| This notebook | `drone-airspace-guardian/ml/colab_continue_train.ipynb` |
| Laptop packer | `drone-airspace-guardian/ml/pack_colab_bundle.py` |
| Live detector | `ml/weights/yolov8n_airspace.pt` loaded by `backend/vision_bridge.py` |

`ml/predict.py` and `ml/model_metadata.json` are the **C-MAPSS health model**. Do not overwrite them. YOLO metrics go to `ml/weights/vision_metrics.json` after you copy exports back.


## 0. What is already on the laptop (warm-start context)

Shipped console weights vs the interrupted mix-train. Colab default is **warm-start from mix `last.pt`**, then run a **new 50-epoch GPU schedule** (`resume=False`). That keeps the 4 mix epochs of filters/head and throws away the 12-epoch CPU cosine (it was the wrong schedule anyway).

### Shipped weights (console today)

| Field | Value |
| --- | --- |
| File | `ml/weights/yolov8n_airspace.pt` (5.94 MB, 2026-09-18 16:32) |
| Model | YOLOv8n, 6 classes: `person, car, van, truck, bus, motor` |
| Data | VisDrone2019-DET native 512 crops (no AU-AIR mix in this export) |
| Epochs / device | **8** / CPU, imgsz 512, batch 8 |
| **mAP50** | **0.2863** (mAP50-95 0.1531, P/R 0.351 / 0.340) |
| Per-class mAP50 | person 0.381, **car 0.627**, van 0.194, truck 0.128, bus 0.107, motor 0.281 |

### Mix-train checkpoints packed in the zip

AU-AIR mixed into **train only** (~20%); **val stays VisDrone-only**. Laptop run died mid epoch 5/12.

| Field | Value |
| --- | --- |
| Completed | **4** epochs (`runs/yolov8n_airspace/results.csv`) |
| Checkpoints | `last.pt` / `best.pt` (23.3 MB, epoch 4) |
| Train / val / holdout | **2231 / 175 / 41** crops at 512×512 |
| Mix | 1785 VisDrone + 446 AU-AIR train |
| In-run val mAP50 | 0.271 → 0.245 → 0.279 → **0.278** |

That dip at epoch 2 is AU-AIR domain mix during warmup, not a dead run. Train cls only moved 1.57 → 1.52 — underfitting, which is why 50 GPU epochs are the job.


## 1. Efficiency this notebook actually implements

| Waste | What the cells do |
| --- | --- |
| Re-upload / Drive unzip of 2k JPEGs | Keep **one zip** on Drive. Skip Drive image extract. Each VM unzips **zip → `/content/airspace`** (SSD, ~1–2 min) and skips if that local tree is already there. |
| `pip` every reconnect | Pin **ultralytics==8.4.155** (laptop checkpoint). Skip install if that version is already loaded. No extra datasets, no CPU torch. |
| CPU runtime | Cell 4 raises **before** Drive mount. |
| `imgsz=640` | **512** — crops are already 512. |
| Tiny batch / no AMP | GPU table **T4 16 / L4 24 / A100 32** (or `BATCH=-1` AutoBatch). **`amp=True`**. |
| Dataloader | **`cache=True` (RAM)** — ~2 GB for 2.2k×512 images, the real speedup. **`workers=2`** (Colab vCPUs; 8 oversubscribe). |
| Plots every epoch | **`plots=False`** during train; curves once after. **`deterministic=False`**. |
| Flat 50 epochs | **`patience=15`**. **`save_period=5`** onto Drive. |
| Schedule | **`cos_lr=True`**, `close_mosaic=10`, `mixup=0.1`. Domain `last.pt` → **`lr0=0.005`**, no freeze. COCO `yolov8n.pt` → **`lr0=0.01`** + 3-epoch backbone freeze then unfreeze. |
| Disconnect / Run all again | Incomplete Drive run → **auto-resume**. Finished run → **skip train**, just export. |
| YOLOv8s | Gated cell, off. |

Checkpoints + `results.csv` live at `MyDrive/airspace-yolo/runs/`. Exports at `MyDrive/airspace-yolo/exports/`.


## 2. Laptop → Drive (once, on Windows)

In PowerShell:

```powershell
cd C:\Users\rohit\Downloads\CodeCortex\drone-airspace-guardian\ml
.\venv\Scripts\python.exe pack_colab_bundle.py
```

Upload **only** the zip (gitignored, ~197 MB). Create the folder in Drive’s UI if needed:

```
My Drive/
  airspace-yolo/
    colab_bundle.zip
```

Do **not** unzip in Drive’s web UI. Do **not** re-download VisDrone or AU-AIR. Class map is locked:

`AUAIR_TO_CLASS = {0:0, 1:1, 2:3, 3:2, 4:5, 6:4, 7:3}` → person, car, van, truck, bus, motor.

If `colab_bundle.zip` is already in that folder from an earlier pack, skip the upload unless you re-packed checkpoints.


## 3. GPU check (fail fast)


In [ ]:
# GPU check - stop here if this prints CPU (do not mount / unzip / pip on a CPU VM)
import torch

print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("gpu", torch.cuda.get_device_name(0))
    print("vram_gb", round(props.total_memory / 1e9, 2))
else:
    raise SystemExit(
        "No GPU. Runtime → Change runtime type → T4 GPU, then Runtime → Restart session."
    )


## 4. Knobs

Default is a **full efficient train**. Change these, then Run all.

| Knob | Default | Meaning |
| --- | --- | --- |
| `MODE` | `"train"` | New 50-epoch job from the start weights. `"resume"` = interrupted **Colab** run only. |
| `START_FROM` | `"auto"` | `last.pt` if the zip has it, else COCO `yolov8n.pt`. Or `"last"` / `"best"` / `"shipped"` / `"coco"` / `"colab_last"`. |
| `EPOCHS` | `50` | New epoch budget in train mode (not added to the old 12). |
| `AUTO_RESUME_IF_INCOMPLETE` | `True` | If Drive already has a partial run, resume it instead of restarting. |
| `SKIP_IF_COMPLETE` | `True` | If that run already hit its epoch target, skip train and export. |


In [ ]:
# ----- knobs (edit here) -----
MODE = "train"  # "train" | "resume"
START_FROM = "auto"  # mix last.pt if packed; extra FT uses Drive best/last
EPOCHS = 80  # GPU job to beat shipped mAP50 0.2863
IMGSZ = 512
CACHE = True  # RAM cache; use "disk" only if Colab RAM is tight
AMP = True
WORKERS = 2  # Colab vCPU; 8 is slower
PLOTS = False  # during train; end cell writes curves once
DETERMINISTIC = False
COS_LR = True
MIXUP = 0.15
COPY_PASTE = 0.3
CLOSE_MOSAIC = 15
PATIENCE = 25
SAVE_PERIOD = 5
SEED = 7
BATCH = 16  # T4
AUTO_RESUME_IF_INCOMPLETE = True
SKIP_IF_COMPLETE = False  # extra epochs if a prior Colab run already finished
FREEZE_EPOCHS = 3  # COCO start only; ignored for domain last.pt
FT_RUN_NAME = "yolov8n_airspace_colab_ft"
FT_EPOCHS = 50
FT_LR0 = 0.0025
ULTRA_PIN = "8.4.155"
RUN_NAME = "yolov8n_airspace_colab"
RUN_S = False  # optional YOLOv8s cell; leave False
# --------------------------------

print(
    dict(
        MODE=MODE,
        START_FROM=START_FROM,
        EPOCHS=EPOCHS,
        IMGSZ=IMGSZ,
        CACHE=CACHE,
        AMP=AMP,
        WORKERS=WORKERS,
        PLOTS=PLOTS,
        BATCH=BATCH,
        PATIENCE=PATIENCE,
        MIXUP=MIXUP,
        COPY_PASTE=COPY_PASTE,
        CLOSE_MOSAIC=CLOSE_MOSAIC,
        SAVE_PERIOD=SAVE_PERIOD,
        AUTO_RESUME_IF_INCOMPLETE=AUTO_RESUME_IF_INCOMPLETE,
        SKIP_IF_COMPLETE=SKIP_IF_COMPLETE,
        FT_RUN_NAME=FT_RUN_NAME,
        FT_EPOCHS=FT_EPOCHS,
        FT_LR0=FT_LR0,
    )
)


## 5. Mount Drive, unzip once, stage to local SSD

First run asks for a Google account. Dataset is **not** copied onto Drive as 2k JPEGs (that stalls the GPU). The zip on Drive is the persistent copy; we unzip it to `/content/airspace` for training. Checkpoints/runs still go to Drive.


In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

from google.colab import drive

if Path("/content/drive/MyDrive").exists():
    print("Drive already mounted — skip OAuth")
else:
    drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/airspace-yolo")
DRIVE.mkdir(parents=True, exist_ok=True)


def _find_zip() -> Path | None:
    roots = [Path("/content/drive/MyDrive"), Path("/content/drive/My Drive")]
    extra = ("airspace-yolo", "airspace_yolo", "airspace", "ml", "CodeCortex")
    cands: list[Path] = []
    for root in roots:
        if not root.exists():
            continue
        cands.append(root / "colab_bundle.zip")
        for d in extra:
            cands.append(root / d / "colab_bundle.zip")
        try:
            for child in root.iterdir():
                if child.is_dir():
                    cands.append(child / "colab_bundle.zip")
        except OSError:
            pass
    seen: set[str] = set()
    for p in cands:
        key = p.as_posix()
        if key in seen:
            continue
        seen.add(key)
        if p.is_file():
            return p
    return None


found = _find_zip()
if found is None:
    root = Path("/content/drive/MyDrive")
    listing = sorted(p.name for p in root.iterdir()) if root.exists() else []
    print("MyDrive top-level:", listing[:50])
    raise SystemExit(
        "colab_bundle.zip not under MyDrive (root or one folder down). "
        "Upload it, then re-run this cell."
    )
ZIP_PATH = DRIVE / "colab_bundle.zip"
if found.resolve() != ZIP_PATH.resolve():
    if not ZIP_PATH.exists():
        print("found zip at", found, "-> copying to", ZIP_PATH)
        shutil.copy2(found, ZIP_PATH)
    else:
        ZIP_PATH = found
        print("using zip at", ZIP_PATH)
else:
    print("zip already at expected path")
LOCAL = Path("/content/airspace")
LOCAL_DATA = LOCAL / "yolo_data"
DRIVE_DATA = DRIVE / "yolo_data"
CKPT = DRIVE / "checkpoints"
RUNS = DRIVE / "runs"
EXPORTS = DRIVE / "exports"
RUNS.mkdir(exist_ok=True)
EXPORTS.mkdir(exist_ok=True)
CKPT.mkdir(exist_ok=True)

print("Drive folder:", DRIVE)
print("zip exists:", ZIP_PATH.exists(), ZIP_PATH)


def _n_jpg(folder: Path) -> int:
    return len(list(folder.glob("*.jpg"))) if folder.exists() else 0


def _train_n(root: Path) -> int:
    return _n_jpg(root / "images" / "train")


expected = 2231
bundle_path = DRIVE / "BUNDLE.json"
if bundle_path.exists():
    expected = int(json.loads(bundle_path.read_text(encoding="utf-8")).get("train_images", expected))

# Do not explode 2k JPEGs onto Drive (that idles the GPU for 10-20 min).
# Zip on Drive is the persistent copy. An older notebook extract is left in place.
if _train_n(DRIVE_DATA) >= expected:
    print("Drive yolo_data already present — skip Drive unzip.")

if _train_n(LOCAL_DATA) >= expected:
    print("Local SSD already staged — skip unzip.")
elif ZIP_PATH.exists():
    print("Unzipping colab_bundle.zip -> /content/airspace (local SSD)...")
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    LOCAL.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(LOCAL)
elif _train_n(DRIVE_DATA) >= 50:
    print("No zip; using Drive yolo_data (first epoch slower until RAM cache fills).")
    LOCAL_DATA = DRIVE_DATA
else:
    raise SystemExit(
        "Upload colab_bundle.zip to My Drive/airspace-yolo/ then re-run this cell."
    )

if _train_n(LOCAL / "yolo_data") >= 50:
    LOCAL_DATA = LOCAL / "yolo_data"
    src_ckpt = LOCAL / "checkpoints"
    if src_ckpt.exists():
        for p in src_ckpt.glob("*.pt"):
            dest = CKPT / p.name
            if not dest.exists():
                shutil.copy2(p, dest)
                print("  checkpoint -> Drive", dest.name)
    src_bundle = LOCAL / "BUNDLE.json"
    if src_bundle.exists() and not (DRIVE / "BUNDLE.json").exists():
        shutil.copy2(src_bundle, DRIVE / "BUNDLE.json")
        bundle_path = DRIVE / "BUNDLE.json"
        expected = int(json.loads(bundle_path.read_text(encoding="utf-8")).get("train_images", expected))

DATA = LOCAL_DATA if _train_n(LOCAL_DATA) >= 50 else DRIVE_DATA
if _train_n(DATA) < 50:
    raise SystemExit(f"No train images under {DATA}")

if bundle_path.exists():
    bundle = json.loads(bundle_path.read_text(encoding="utf-8"))
    expected = int(bundle.get("train_images", expected))
    print("bundle:", bundle.get("note", "")[:160])

for split in ("train", "val", "auair_holdout"):
    n_img = _n_jpg(DATA / "images" / split)
    n_lbl = len(list((DATA / "labels" / split).glob("*.txt"))) if (DATA / "labels" / split).exists() else 0
    print(f"  {split:14s} images={n_img:4d}  labels={n_lbl:4d}")
    if split == "train" and n_img < expected:
        raise SystemExit(
            f"Train extract incomplete ({n_img} < {expected}). "
            "Runtime -> Restart session, or rm -rf /content/airspace and re-run this cell."
        )

print("checkpoints on Drive:", sorted(p.name for p in CKPT.glob("*.pt")))
print("DATA", DATA)


## 6. Point `data.yaml` at the local dataset. Class **names and ids stay identical** so `last.pt` stays valid.


In [ ]:
NAMES = ["person", "car", "van", "truck", "bus", "motor"]


def write_yaml(path: Path, train_rel: str, val_rel: str) -> None:
    lines = [
        f"path: {DATA.as_posix()}",
        f"train: {train_rel}",
        f"val: {val_rel}",
        "names:",
        *[f"  {i}: {n}" for i, n in enumerate(NAMES)],
        "",
    ]
    path.write_text(chr(10).join(lines), encoding="utf-8")


YAML = DATA / "data.yaml"
HOLDOUT_YAML = DATA / "auair_holdout.yaml"
write_yaml(YAML, "images/train", "images/val")
if (DATA / "images" / "auair_holdout").exists():
    write_yaml(HOLDOUT_YAML, "images/train", "images/auair_holdout")

print(YAML.read_text())


## 7. Install ultralytics **8.4.155** only (matches laptop `last.pt`). Colab already has CUDA torch — do not pip-install torch.


In [ ]:
import importlib.metadata
import subprocess
import sys

def _ultra_ver() -> str | None:
    try:
        return importlib.metadata.version("ultralytics")
    except importlib.metadata.PackageNotFoundError:
        return None

have = _ultra_ver()
if have != ULTRA_PIN:
    print(f"installing ultralytics=={ULTRA_PIN} (have {have})")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRA_PIN}"]
    )
else:
    print(f"ultralytics {have} already pinned - skip pip")

import os

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

import ultralytics
try:
    from ultralytics.utils import SETTINGS

    SETTINGS.update({"wandb": False, "tensorboard": False, "mlflow": False})
except Exception as exc:
    print("ultralytics SETTINGS skip:", exc)
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())


## 8. Train

Writes under `MyDrive/airspace-yolo/runs/yolov8n_airspace_colab/`.

Disconnect: remount + Run all. With the defaults, an incomplete Drive run **resumes**; a finished one **exports only**. To force a new 50 from current weights, set `MODE="train"`, `AUTO_RESUME_IF_INCOMPLETE=False`, `SKIP_IF_COMPLETE=False`, `START_FROM="colab_last"`.


In [ ]:
import os
import time

from ultralytics import YOLO

os.environ["WANDB_DISABLED"] = "true"


def pick_batch() -> int:
    if BATCH == -1:
        return -1
    if BATCH > 0:
        return int(BATCH)
    name = torch.cuda.get_device_name(0).upper()
    if "A100" in name:
        return 32
    if "L4" in name or "V100" in name:
        return 24
    if "T4" in name:
        return 16
    return 8


def csv_epochs(run_dir: Path) -> int:
    csv_path = run_dir / "results.csv"
    if not csv_path.exists():
        return 0
    rows = csv_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    return max(0, len(rows) - 1)


def planned_epochs(run_dir: Path, fallback: int) -> int:
    args = run_dir / "args.yaml"
    if not args.exists():
        return fallback
    for line in args.read_text(encoding="utf-8", errors="ignore").splitlines():
        if line.startswith("epochs:"):
            try:
                return int(line.split(":", 1)[1].strip())
            except ValueError:
                return fallback
    return fallback


def resolve_start() -> tuple[Path | str, str, bool]:
    starts = {
        "last": CKPT / "last.pt",
        "best": CKPT / "best.pt",
        "shipped": CKPT / "yolov8n_airspace.pt",
        "colab_last": RUNS / RUN_NAME / "weights" / "last.pt",
        "coco": "yolov8n.pt",
    }
    key = START_FROM
    if key == "auto":
        key = "last" if starts["last"].exists() else "coco"
    if key == "coco":
        return "yolov8n.pt", "coco", True
    path = starts[key]
    if isinstance(path, Path) and not path.exists():
        raise SystemExit(f"Missing {path}. Re-pack the zip or set START_FROM='coco'.")
    return path, key, False


BATCH_EFF = pick_batch()
start_pt, start_key, from_coco = resolve_start()
run_dir = RUNS / RUN_NAME
colab_last = run_dir / "weights" / "last.pt"
done_ep = csv_epochs(run_dir)
plan_ep = planned_epochs(run_dir, EPOCHS)

mode_eff = MODE
SKIPPED_TRAIN = False
epochs_eff = EPOCHS
LR0_OVERRIDE = None

if MODE == "resume":
    if not colab_last.exists():
        raise SystemExit(
            f"MODE=resume needs {colab_last}. Set MODE='train' for a fresh GPU job."
        )
    start_pt, start_key, from_coco = colab_last, "colab_last", False
elif AUTO_RESUME_IF_INCOMPLETE and colab_last.exists() and done_ep < plan_ep:
    mode_eff = "resume"
    start_pt, start_key, from_coco = colab_last, "colab_last", False
    print(f"auto-resume incomplete Colab run ({done_ep}/{plan_ep})")
elif (not SKIP_IF_COMPLETE) and colab_last.exists() and done_ep >= plan_ep and plan_ep > 0:
    ft_best = run_dir / "weights" / "best.pt"
    ft_src = ft_best if ft_best.exists() else colab_last
    RUN_NAME = FT_RUN_NAME
    run_dir = RUNS / RUN_NAME
    start_pt, start_key, from_coco = ft_src, "colab_best", False
    epochs_eff = FT_EPOCHS
    LR0_OVERRIDE = FT_LR0
    print(f"finished Drive run {done_ep}/{plan_ep} — extra FT {epochs_eff} from {ft_src} as {RUN_NAME}")
elif SKIP_IF_COMPLETE and colab_last.exists() and done_ep >= plan_ep and plan_ep > 0:
    SKIPPED_TRAIN = True
    print(f"skip train: {done_ep}/{plan_ep} epochs already on Drive")

# Domain last.pt: lower LR, no freeze. COCO: default LR + short freeze then unfreeze.
if from_coco:
    LR0 = 0.01
    WARMUP_EPOCHS = 3.0
    do_freeze = FREEZE_EPOCHS > 0 and mode_eff == "train"
else:
    LR0 = FT_LR0 if LR0_OVERRIDE is not None else 0.005
    WARMUP_EPOCHS = 1.0
    do_freeze = False

print(
    dict(
        mode_eff=mode_eff,
        start=str(start_pt),
        from_coco=from_coco,
        batch=BATCH_EFF,
        lr0=LR0,
        freeze_stage=do_freeze,
        device=torch.cuda.get_device_name(0),
        cpu_count=os.cpu_count(),
        workers=WORKERS,
        cache=CACHE,
        amp=AMP,
    )
)


def train_kwargs(**extra):
    kw = dict(
        data=str(YAML),
        imgsz=IMGSZ,
        batch=BATCH_EFF,
        device=0,
        workers=WORKERS,
        project=str(RUNS),
        name=RUN_NAME,
        exist_ok=True,
        resume=False,
        pretrained=True,
        patience=PATIENCE,
        cos_lr=COS_LR,
        close_mosaic=CLOSE_MOSAIC,
        mixup=MIXUP,
        copy_paste=COPY_PASTE,
        lr0=LR0,
        warmup_epochs=WARMUP_EPOCHS,
        save=True,
        save_period=SAVE_PERIOD,
        val=True,
        plots=PLOTS,
        verbose=True,
        seed=SEED,
        amp=AMP,
        cache=CACHE,
        deterministic=DETERMINISTIC,
        freeze=None,
    )
    kw.update(extra)
    return kw


def train_with_oom_backoff(model, kwargs):
    global BATCH_EFF
    batch = kwargs["batch"]
    while True:
        try:
            return model.train(**kwargs)
        except RuntimeError as exc:
            msg = str(exc).lower()
            if "out of memory" not in msg or not isinstance(batch, int) or batch <= 4 or batch == -1:
                raise
            torch.cuda.empty_cache()
            batch = max(4, batch // 2)
            kwargs["batch"] = batch
            BATCH_EFF = batch
            print(f"CUDA OOM — retry batch={batch}")


elapsed = 0.0
results = None
started = time.time()

if SKIPPED_TRAIN:
    pass
elif mode_eff == "resume":
    model = YOLO(str(colab_last))
    results = model.train(resume=True)
else:
    model = YOLO(str(start_pt))
    if do_freeze:
        s1 = min(FREEZE_EPOCHS, EPOCHS)
        s2 = max(1, EPOCHS - s1)
        print(f"stage 1: freeze backbone {s1} epochs, then unfreeze {s2}")
        train_with_oom_backoff(
            model,
            train_kwargs(
                epochs=s1,
                freeze=10,
                close_mosaic=0,
                patience=s1,
                save_period=1,
            ),
        )
        frozen = run_dir / "weights" / "last.pt"
        hold = CKPT / "frozen_stage1.pt"
        shutil.copy2(frozen, hold)
        model = YOLO(str(hold))
        results = train_with_oom_backoff(
            model, train_kwargs(epochs=s2, freeze=None)
        )
    else:
        results = train_with_oom_backoff(
            model, train_kwargs(epochs=epochs_eff, freeze=None)
        )

elapsed = time.time() - started
print(f"train wall {elapsed/60:.1f} min  skip={SKIPPED_TRAIN}")
print("run dir", run_dir)


## 9. Curves once, re-validate `best.pt`, export to Drive. VisDrone val is the headline number; AU-AIR holdout is recorded separately.


In [ ]:
from datetime import datetime

run_dir = RUNS / RUN_NAME
best = run_dir / "weights" / "best.pt"
last = run_dir / "weights" / "last.pt"
src = best if best.exists() else last
if not src.exists():
    raise SystemExit(f"No weights under {run_dir / 'weights'}")

csv_path = run_dir / "results.csv"
if csv_path.exists():
    try:
        from ultralytics.utils.plotting import plot_results

        plot_results(file=str(csv_path), dir=str(run_dir))
        print("wrote train curves next to results.csv")
    except Exception as exc:
        print("plot_results skipped:", exc)

exported_path = EXPORTS / "yolov8n_airspace.pt"
shutil.copy2(src, exported_path)
print("copied", src, "->", exported_path)

exported = YOLO(str(exported_path))


def box_metrics(final) -> dict:
    box = final.box
    return {
        "mAP50": round(float(box.map50), 4),
        "mAP50_95": round(float(box.map), 4),
        "precision": round(float(box.mp), 4),
        "recall": round(float(box.mr), 4),
        "per_class_mAP50": {
            NAMES[int(c)]: round(float(box.ap50[i]), 4)
            for i, c in enumerate(final.box.ap_class_index)
            if int(c) < len(NAMES)
        },
    }


visdrone = box_metrics(
    exported.val(
        data=str(YAML),
        imgsz=IMGSZ,
        batch=max(1, BATCH_EFF if BATCH_EFF > 0 else 16),
        device=0,
        workers=WORKERS,
        verbose=False,
        plots=True,
    )
)
print("VisDrone val", visdrone)

auair_hold = None
n_hold = len(list((DATA / "images" / "auair_holdout").glob("*.jpg"))) if (DATA / "images" / "auair_holdout").exists() else 0
if HOLDOUT_YAML.exists() and n_hold >= 10:
    auair_hold = box_metrics(
        exported.val(
            data=str(HOLDOUT_YAML),
            imgsz=IMGSZ,
            batch=max(1, BATCH_EFF if BATCH_EFF > 0 else 16),
            device=0,
            workers=WORKERS,
            verbose=False,
            plots=False,
        )
    )
    print("AU-AIR holdout", auair_hold, "on", n_hold, "crops")

n_train = len(list((DATA / "images" / "train").glob("*.jpg")))
n_val = len(list((DATA / "images" / "val").glob("*.jpg")))
actual_epochs = csv_epochs(run_dir)
metrics = {
    "model": "yolov8n-airspace",
    "epochs": actual_epochs or EPOCHS,
    "images": n_train,
    "mAP50": visdrone["mAP50"],
    "mAP50_95": visdrone["mAP50_95"],
    "precision": visdrone["precision"],
    "recall": visdrone["recall"],
    "imgsz": IMGSZ,
    "crop": 512,
    "classes": NAMES,
    "per_class_mAP50": visdrone["per_class_mAP50"],
    "val_images": n_val,
    "train_seconds": round(elapsed, 1),
    "dataset": "VisDrone2019-DET-train native 512 crops + AU-AIR train mix; val is VisDrone-only",
    "auair_holdout_crops": n_hold,
    "auair_holdout_mAP50": None if auair_hold is None else auair_hold["mAP50"],
    "auair_holdout": auair_hold,
    "baseline_mAP50": 0.2863,
    "beat_baseline": visdrone["mAP50"] > 0.2863,
    "device": torch.cuda.get_device_name(0),
    "batch": BATCH_EFF,
    "cache": CACHE,
    "amp": AMP,
    "workers": WORKERS,
    "start_weights": str(getattr(start_pt, "name", start_pt)),
    "mode": mode_eff,
    "skipped_train": SKIPPED_TRAIN,
    "trained_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "colab": True,
}
metrics_path = EXPORTS / "vision_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
if csv_path.exists():
    shutil.copy2(csv_path, EXPORTS / "results.csv")
print("=" * 72)
print(f"  VisDrone val mAP50  {metrics['mAP50']:.4f}  (baseline 0.2863)")
print(f"  mAP50-95            {metrics['mAP50_95']:.4f}")
print(f"  beat_baseline       {metrics['beat_baseline']}")
print(f"  metrics -> {metrics_path}")
print("=" * 72)


## 10. Optional YOLOv8s (off)

Skip unless n finished and VisDrone mAP50 is still **< 0.32** with a flat `results.csv`. s cannot load n weights. Do not run this in parallel with the n train cell on a free T4.


In [ ]:
if RUN_S:
    s_model = YOLO("yolov8s.pt")
    s_model.train(
        data=str(YAML),
        epochs=40,
        imgsz=IMGSZ,
        batch=max(8, (BATCH_EFF // 2) if BATCH_EFF > 0 else 8),
        device=0,
        workers=WORKERS,
        project=str(RUNS),
        name="yolov8s_airspace_colab",
        exist_ok=True,
        patience=12,
        cos_lr=True,
        close_mosaic=10,
        mixup=0.1,
        lr0=0.01,
        seed=SEED,
        amp=True,
        cache=CACHE,
        plots=False,
        deterministic=False,
    )
    s_best = RUNS / "yolov8s_airspace_colab" / "weights" / "best.pt"
    print("s best:", s_best)
else:
    print("skipped yolov8s (RUN_S is False)")


## 11. Copy weights back to the laptop

From Drive, download `MyDrive/airspace-yolo/exports/`:

- `yolov8n_airspace.pt`
- `vision_metrics.json`

On the laptop:

```powershell
$src = "$env:USERPROFILE\Downloads"
$dst = "C:\Users\rohit\Downloads\CodeCortex\drone-airspace-guardian\ml\weights"
Copy-Item "$src\yolov8n_airspace.pt"  "$dst\yolov8n_airspace.pt" -Force
Copy-Item "$src\vision_metrics.json"  "$dst\vision_metrics.json" -Force
```

`backend/vision_bridge.py` already loads `ml/weights/yolov8n_airspace.pt`. Restart uvicorn.

Do **not** copy over `ml/model_metadata.json` or `ml/rul_model.joblib`. Do **not** edit `predict.py`.

Optional keep: `MyDrive/airspace-yolo/runs/yolov8n_airspace_colab/weights/best.pt` and `results.csv`.


In [ ]:
from google.colab import files

print("exports:")
for p in sorted(EXPORTS.iterdir()):
    print(f"  {p.name:24s} {p.stat().st_size/1e6:.2f} MB")

files.download(str(EXPORTS / "yolov8n_airspace.pt"))
files.download(str(EXPORTS / "vision_metrics.json"))


## 12. If something fails

| Symptom | Fix |
| --- | --- |
| `No GPU` | Runtime type → GPU, then **Restart session** |
| zip not found | `My Drive/airspace-yolo/colab_bundle.zip` exactly |
| Local unzip incomplete | `rm -rf /content/airspace` and re-run the mount cell |
| `last.pt` missing | Re-run `pack_colab_bundle.py`; or `START_FROM="coco"` |
| CUDA OOM | Train cell halves batch; or set `BATCH=8` in knobs |
| `resume` FileNotFound on `C:\Users\...` | Laptop checkpoint is not a Colab run. `MODE="train"` |
| Disconnect mid-epoch | Run all again. Auto-resume uses Drive `last.pt` |
| mAP still ~0.28 after 50 | Then `RUN_S = True`, or re-pack with `YOLO_SRC_TRAIN=2000` |
| Class count mismatch | Do not edit `names:`; 6 classes in this order |
